# Retail Inventory Management System
## Notebook 3 — Gold Layer
### Purpose
This notebook reads from Silver layer tables and generates business ready Gold tables.
SQL joins and aggregations are used to detect low stock situations, overstock situations,
and produce category wise stock summaries for reporting and alerting.

#### Importing required libraries

In [0]:
# Cell 1 - setup
spark.sql("USE CATALOG retail_inventory")

import logging
from pyspark.sql.functions import count, sum, avg, min, max


#### Creating Fact and Dimention Tables 

In [0]:
%sql
CREATE OR REPLACE TABLE retail_inventory.gold.fact_inventory AS
SELECT
    id AS product_id,
    store_id,
    stock,
    low_stock_threshold,
    overstock_threshold,
    load_date
FROM retail_inventory.silver.inventory_by_store;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE retail_inventory.gold.dim_products AS
SELECT DISTINCT
    id AS product_id,
    product_name,
    category,
    brand
FROM retail_inventory.silver.products;

num_affected_rows,num_inserted_rows


####  Low Stock Alerts

In [0]:

try:
    spark.sql("""
        CREATE OR REPLACE TABLE retail_inventory.gold.low_stock_alerts AS
        SELECT
            p.id,
            p.product_name,
            p.category,
            p.brand,
            i.store_id,
            i.stock               AS quantity,
            i.low_stock_threshold,
            current_date()        AS alert_date,
            'LOW STOCK'           AS alert_type
        FROM retail_inventory.silver.inventory_by_store i
        JOIN retail_inventory.silver.products p
          ON i.id = p.id
        WHERE i.stock < i.low_stock_threshold
        ORDER BY i.stock ASC
    """)

    low_count = spark.table(
        "retail_inventory.gold.low_stock_alerts"
    ).count()
    print(f"Low stock alerts: {low_count}")

except Exception as e:
    logger.error(f"Failed to create low stock alerts: {e}")
    raise

Low stock alerts: 21


####  Overstock Alerts

In [0]:

try:
    spark.sql("""
        CREATE OR REPLACE TABLE retail_inventory.gold.overstock_alerts AS
        SELECT
            p.id,
            p.product_name,
            p.category,
            p.brand,
            i.store_id,
            i.stock               AS quantity,
            i.overstock_threshold,
            current_date()        AS alert_date,
            'OVERSTOCK'           AS alert_type
        FROM retail_inventory.silver.inventory_by_store i
        JOIN retail_inventory.silver.products p
          ON i.id = p.id
        WHERE i.stock > i.overstock_threshold
        ORDER BY i.stock DESC
    """)

    high_count = spark.table(
        "retail_inventory.gold.overstock_alerts"
    ).count()
    print(f"Overstock alerts: {high_count}")

except Exception as e:
    logger.error(f"Failed to create overstock alerts: {e}")
    raise

Overstock alerts: 10


#### Category Stock Summary

In [0]:

# Cell 4 - category summary aggregation
spark.sql("""
    CREATE OR REPLACE TABLE retail_inventory.gold.category_stock_summary AS
    SELECT
        p.category,
        COUNT(p.id)    AS total_products,
        SUM(i.stock)   AS total_stock,
        AVG(i.stock)   AS avg_stock,
        MIN(i.stock)   AS min_stock,
        MAX(i.stock)   AS max_stock,
        current_date() AS report_date
    FROM retail_inventory.silver.inventory_by_store i
    JOIN retail_inventory.silver.products p
      ON i.id = p.id
    GROUP BY p.category
    ORDER BY total_stock ASC
""")

print("Category summary created")

#

Category summary created


#### Verify category_stock_summary

In [0]:
%sql
select* from retail_inventory.gold.category_stock_summary;

category,total_products,total_stock,avg_stock,min_stock,max_stock,report_date
furniture,15,462,30.8,2,88,2026-04-07
fragrances,15,527,35.13333333333333,1,98,2026-04-07
beauty,15,740,49.333333333333336,10,99,2026-04-07
groceries,45,1376,30.57777777777778,0,99,2026-04-07


#### Verify low_stock_alerts

In [0]:
%sql
select* from retail_inventory.gold.low_stock_alerts
limit 5

id,product_name,category,brand,store_id,quantity,low_stock_threshold,alert_date,alert_type
26,Green Chili Pepper,groceries,null,3,0,10,2026-04-07,LOW STOCK
9,Dolce Shine Eau de,fragrances,Dolce & Gabbana,3,1,10,2026-04-07,LOW STOCK
26,Green Chili Pepper,groceries,null,2,1,10,2026-04-07,LOW STOCK
9,Dolce Shine Eau de,fragrances,Dolce & Gabbana,2,2,10,2026-04-07,LOW STOCK
15,Wooden Bathroom Sink With Mirror,furniture,Bath Trends,3,2,10,2026-04-07,LOW STOCK


#### Verify overstock_alerts

In [0]:
%sql
select* from retail_inventory.gold.overstock_alerts
limit 5

id,product_name,category,brand,store_id,quantity,overstock_threshold,alert_date,alert_type
1,Essence Mascara Lash Princess,beauty,Essence,1,99,80,2026-04-07,OVERSTOCK
30,Kiwi,groceries,null,1,99,80,2026-04-07,OVERSTOCK
8,Dior J'adore,fragrances,Dior,1,98,80,2026-04-07,OVERSTOCK
19,Chicken Meat,groceries,null,1,97,80,2026-04-07,OVERSTOCK
4,Red Lipstick,beauty,Chic Cosmetics,1,91,80,2026-04-07,OVERSTOCK
